# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on kaggle — storage at: /kaggle/working
Running on kaggle — storage at: /kaggle/working


/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

optimizer = ModelOptimizer("RP3beta")

STUDY_NAME = RP3betaRecommender.RECOMMENDER_NAME

In [20]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "alpha": optuna_trial.suggest_float("alpha", 0.5, 2.0),
        "beta": optuna_trial.suggest_float("beta", 0.0, 1.0),
        "topK": optuna_trial.suggest_int("topK", 100, 1000),
        "implicit": True
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = RP3betaRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [21]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-21 11:21:07,417] A new study created in RDB with name: RP3betaRecommender


  0%|          | 0/100 [00:00<?, ?it/s]

RP3betaRecommender: Similarity column 6969 (100.0%), 653.48 column/sec. Elapsed time 10.66 sec
  Fold 1/5 - Score: 0.20668547734357062
RP3betaRecommender: Similarity column 6969 (100.0%), 652.65 column/sec. Elapsed time 10.68 sec
  Fold 2/5 - Score: 0.20776049042029957
RP3betaRecommender: Similarity column 6969 (100.0%), 650.17 column/sec. Elapsed time 10.72 sec
  Fold 3/5 - Score: 0.2079358017834494
RP3betaRecommender: Similarity column 6969 (100.0%), 653.58 column/sec. Elapsed time 10.66 sec
  Fold 4/5 - Score: 0.2064264698597831
RP3betaRecommender: Similarity column 6969 (100.0%), 659.59 column/sec. Elapsed time 10.57 sec
  Fold 5/5 - Score: 0.20745748801070124
[I 2025-11-21 11:23:52,690] Trial 0 finished with value: 0.2072531454835608 and parameters: {'alpha': 1.0842068457692469, 'beta': 0.2914418760346168, 'topK': 644}. Best is trial 0 with value: 0.2072531454835608.
RP3betaRecommender: Similarity column 6969 (100.0%), 853.85 column/sec. Elapsed time 8.16 sec
  Fold 1/5 - Score: 0

In [10]:
optuna_study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=paths.OPTUNA_STORAGE
)

In [11]:
optuna.visualization.plot_optimization_history(optuna_study)

In [12]:
optuna.visualization.plot_param_importances(optuna_study)

In [13]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [14]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "alpha": optuna_trial.suggest_float("alpha", 1.5, 1.7),
        "beta": optuna_trial.suggest_float("beta", 0.28, 0.32),
        "topK": optuna_trial.suggest_int("topK", 50, 110),
        "implicit": True
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = RP3betaRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [15]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-21 18:03:43,380] A new study created in RDB with name: RP3betaRecommender_refined


  0%|          | 0/20 [00:00<?, ?it/s]

RP3betaRecommender: Similarity column 6969 (100.0%), 1520.88 column/sec. Elapsed time 4.58 sec
  Fold 1/5 - Score: 0.24703659414234114
RP3betaRecommender: Similarity column 6969 (100.0%), 1534.87 column/sec. Elapsed time 4.54 sec
  Fold 2/5 - Score: 0.24602120585449003
RP3betaRecommender: Similarity column 6969 (100.0%), 1550.67 column/sec. Elapsed time 4.49 sec
  Fold 3/5 - Score: 0.2468555330334698
RP3betaRecommender: Similarity column 6969 (100.0%), 1509.10 column/sec. Elapsed time 4.62 sec
  Fold 4/5 - Score: 0.2458011260772326
RP3betaRecommender: Similarity column 6969 (100.0%), 1533.69 column/sec. Elapsed time 4.54 sec
  Fold 5/5 - Score: 0.24810313868095477
[I 2025-11-21 18:05:13,732] Trial 0 finished with value: 0.24676351955769765 and parameters: {'alpha': 1.5863533346216319, 'beta': 0.3133195613531605, 'topK': 86}. Best is trial 0 with value: 0.24676351955769765.
RP3betaRecommender: Similarity column 6969 (100.0%), 1520.15 column/sec. Elapsed time 4.58 sec
  Fold 1/5 - Score:

In [16]:
optuna.visualization.plot_optimization_history(optuna_study)

In [17]:
optuna.visualization.plot_param_importances(optuna_study)

In [18]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [20]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "alpha": optuna_trial.suggest_float("alpha", 1.55, 1.64),
        "beta": optuna_trial.suggest_float("beta", 0.29, 0.32),
        "topK": optuna_trial.suggest_int("topK", 20, 60),
        "implicit": True
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = RP3betaRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [21]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined_2",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-21 18:51:01,508] A new study created in RDB with name: RP3betaRecommender_refined_2


  0%|          | 0/20 [00:00<?, ?it/s]

RP3betaRecommender: Similarity column 6969 (100.0%), 1694.17 column/sec. Elapsed time 4.11 sec
  Fold 1/5 - Score: 0.24816230958326438
RP3betaRecommender: Similarity column 6969 (100.0%), 1700.29 column/sec. Elapsed time 4.10 sec
  Fold 2/5 - Score: 0.247306215787961
RP3betaRecommender: Similarity column 6969 (100.0%), 1647.07 column/sec. Elapsed time 4.23 sec
  Fold 3/5 - Score: 0.24777696220185882
RP3betaRecommender: Similarity column 6969 (100.0%), 1688.20 column/sec. Elapsed time 4.13 sec
  Fold 4/5 - Score: 0.2472086221135018
RP3betaRecommender: Similarity column 6969 (100.0%), 1659.25 column/sec. Elapsed time 4.20 sec
  Fold 5/5 - Score: 0.248474023241323
[I 2025-11-21 18:52:25,652] Trial 0 finished with value: 0.2477856265855818 and parameters: {'alpha': 1.6070734121996404, 'beta': 0.2925576980229501, 'topK': 51}. Best is trial 0 with value: 0.2477856265855818.
RP3betaRecommender: Similarity column 6969 (100.0%), 1658.31 column/sec. Elapsed time 4.20 sec
  Fold 1/5 - Score: 0.24

In [22]:
optuna.visualization.plot_optimization_history(optuna_study)

In [23]:
optuna.visualization.plot_param_importances(optuna_study)

In [24]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Trial 1:
Best Value: 0.24607393690050236
Best Params: {'alpha': 1.6210374000273637, 'beta': 0.3024355866585492, 'topK': 100}

- Trial 2:
Best Value: 0.2479890934816337
Best Params: {'alpha': 1.6063752805768927, 'beta': 0.30796686246709554, 'topK': 57}

- Trial 3:
Best Value: 0.24848583049324713
Best Params: {'alpha': 1.5509591048362328, 'beta': 0.30751121269314674, 'topK': 39}

Best Value: 0.24848583049324713


Best Params: {'alpha': 1.5509591048362328, 'beta': 0.30751121269314674, 'topK': 39}